<a href="https://colab.research.google.com/github/amorimriki/Projeto-I-PSA/blob/main/JUPYTER_NOTEBOOK/model_for_GoogleColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# --------------------------------------------------------------
# Dependencies
# --------------------------------------------------------------
!pip install statsmodels

import pandas as pd
import pylab as pl
import numpy as np
import scipy.optimize as opt
import statsmodels.api as sm

from google.colab import files


import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
import seaborn as sns


import itertools

import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix




In [3]:
# --------------------------------------------------------------
# Import Dataset
# --------------------------------------------------------------


df = pd.read_csv("/content/student_data_merge.csv")



In [4]:
# =====================
# Features + Preprocessamento + Split
# =====================



# Features e target
target = 'final_result'
X = df.drop(columns=[target])
y = df[target]

# Features categóricas e numéricas
categorical_features = ['code_module', 'gender', 'region', 'highest_education',
                        'imd_band', 'age_band', 'disability', 'assessment_type', 'is_banked']
numerical_features = ['date_submitted', 'num_of_prev_attempts', 'sum_click',
                      'date', 'studied_credits', 'weight', 'score']

# Preprocessamento
categorical_transformer = OneHotEncoder(handle_unknown='ignore')
numerical_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', numerical_transformer, numerical_features)
    ]
)

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
'''# =====================
# Random Forest Pipeline + GridSearchCV
# =====================
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=3, scoring='accuracy', n_jobs=-1, verbose=3)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
print("Melhor Random Forest:")
print(grid_rf.best_params_)
print(classification_report(y_test, best_rf.predict(X_test)))

# Guardar modelo
joblib.dump(best_rf, 'rf_pipeline.pkl')
files.download('rf_pipeline.pkl')'''


# =====================
# SVM Pipeline + GridSearchCV
# =====================
# Demasiado Lento + de 10 horas a treinar e foi cancelado

svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True))
])

param_grid_svm = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(svm_pipeline, param_grid_svm, cv=3, scoring='accuracy', n_jobs=-1, verbose=3)
grid_svm.fit(X_train, y_train)
best_svm = grid_svm.best_estimator_
print("Melhor SVM:")
print(grid_svm.best_params_)
print(classification_report(y_test, best_svm.predict(X_test)))

# Guardar modelo
joblib.dump(best_svm, 'svm_pipeline.pkl')
files.download('svm_pipeline.pkl')



In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# =====================
# SVM Pipeline + RandomizedSearchCV
# =====================
svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True))
])

param_dist_svm = {
    'classifier__C': uniform(0.1, 10),  # distribuição contínua entre 0.1 e 10.1
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__gamma': ['scale', 'auto']
}

random_svm = RandomizedSearchCV(
    svm_pipeline,
    param_distributions=param_dist_svm,
    n_iter=10,  # número de combinações a testar
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=3,
    random_state=42
)

random_svm.fit(X_train, y_train)
best_svm = random_svm.best_estimator_

print("Melhor SVM:")
print(random_svm.best_params_)
print(classification_report(y_test, best_svm.predict(X_test)))

# Guardar modelo
joblib.dump(best_svm, 'svm_pipeline.pkl')
files.download('svm_pipeline.pkl')


Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [ ]:
# =====================
# MLP Pipeline + GridSearchCV
# =====================
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', MLPClassifier(max_iter=300, random_state=42))
])

param_grid_mlp = {
    'classifier__hidden_layer_sizes': [(50,), (100,), (100, 50)],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__solver': ['adam'],
    'classifier__alpha': [0.0001, 0.001],
    'classifier__learning_rate': ['constant', 'adaptive']
}

grid_mlp = GridSearchCV(mlp_pipeline, param_grid_mlp, cv=3, scoring='accuracy', n_jobs=-1, verbose=3)
grid_mlp.fit(X_train, y_train)
best_mlp = grid_mlp.best_estimator_
print("Melhor MLP:")
print(grid_mlp.best_params_)
print(classification_report(y_test, best_mlp.predict(X_test)))

# Guardar modelo
joblib.dump(best_mlp, 'mlp_pipeline.pkl')
files.download('mlp_pipeline.pkl')


Fitting 3 folds for each of 24 candidates, totalling 72 fits
Melhor MLP:
{'classifier__activation': 'tanh', 'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate': 'constant', 'classifier__solver': 'adam'}
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      6808
           1       0.98      0.99      0.99     29701

    accuracy                           0.98     36509
   macro avg       0.97      0.96      0.96     36509
weighted avg       0.98      0.98      0.98     36509



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# =====================
# SVM Pipeline + HalvingRandomSearchCV
# =====================

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.experimental import enable_halving_search_cv
from sklearn.metrics import classification_report
from scipy.stats import uniform
import joblib
from google.colab import files  # Se estiver usando Colab

# Criar pipeline
svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True))
])
# --- Treinar pipeline sem tuning ---
svm_pipeline.fit(X_train, y_train)

# --- Salvar versão inicial treinada (sem otimização) ---
joblib.dump(svm_pipeline, 'svm_pipeline_nao_otimizado.pkl')
files.download('svm_pipeline_nao_otimizado.pkl')





<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.experimental import enable_halving_search_cv  # apenas para habilitar
from sklearn.model_selection import HalvingRandomSearchCV   # aqui você importa o que vai usar
from sklearn.metrics import classification_report
from scipy.stats import uniform
import joblib
from google.colab import files

svm_pipeline = joblib.load('/content/svm_pipeline_nao_otimizado.pkl')

# Espaço de busca para hiperparâmetros
param_dist_svm = {
    'classifier__C': uniform(0.1, 10),
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__gamma': ['scale', 'auto']
}

# Configurar HalvingRandomSearchCV
halving_random_svm = HalvingRandomSearchCV(
    svm_pipeline,
    param_distributions=param_dist_svm,
    factor=3,             # Reduz a cada 3x (mais rápido que factor=2)
    resource='n_samples',
    max_resources='auto',
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=3,
    n_candidates=200      # Só começa com 200 combinações aleatórias
)

# Treinar com busca de hiperparâmetros
halving_random_svm.fit(X_train, y_train)

# Melhor modelo encontrado
best_svm = halving_random_svm.best_estimator_

print("Melhor SVM:")
print(halving_random_svm.best_params_)
print(classification_report(y_test, best_svm.predict(X_test)))

# --- Salvar versão otimizada (com tuning) ---
joblib.dump(best_svm, 'svm_pipeline_otimizado.pkl')
files.download('svm_pipeline_otimizado.pkl')


n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 9
min_resources_: 20
max_resources_: 146035
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 200
n_resources: 20
Fitting 5 folds for each of 200 candidates, totalling 1000 fits
----------
iter: 1
n_candidates: 67
n_resources: 60
Fitting 5 folds for each of 67 candidates, totalling 335 fits
----------
iter: 2
n_candidates: 23
n_resources: 180
Fitting 5 folds for each of 23 candidates, totalling 115 fits
----------
iter: 3
n_candidates: 8
n_resources: 540
Fitting 5 folds for each of 8 candidates, totalling 40 fits
----------
iter: 4
n_candidates: 3
n_resources: 1620
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Melhor SVM:
{'classifier__C': np.float64(4.085047343973734), 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
              precision    recall  f1-score   support

           0       0.79      0.24      0.37      6808
           1       0.85      0.99      0.91     29701

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =====================
# Ensemble com VotingClassifier
# =====================
ensemble = VotingClassifier(
    estimators=[
        ('rf', best_rf),
        ('svm', best_svm),
        ('mlp', best_mlp)
    ],
    voting='soft'
)

ensemble.fit(X_train, y_train)
y_pred_ensemble = ensemble.predict(X_test)

print("Avaliação do Ensemble:")
print(classification_report(y_test, y_pred_ensemble))
print(confusion_matrix(y_test, y_pred_ensemble))

# Guardar modelo ensemble
joblib.dump(ensemble, 'ensemble_model.pkl')
files.download('ensemble_model.pkl')
